# 02: Building a Baseline Model (TF-IDF + Logistic Regression)
This notebook is dedicated to creating a baseline model for classifying financial intents. The aim is to establish an initial benchmark against which we will compare more complex architectures.

### 1. Evaluation strategy and metrics:
**Primary metric:** Macro F1-score (designed for imbalanced classes; treats the quality of each of the 77 intents equally).

**Supplementary metrics:** Weighted F1-score, ROC-AUC (One-vs-Rest)`

In [1]:
# Automatic reloading of modules whilst editing
%load_ext autoreload
%autoreload 2

# Setting up the runtime environment and importing necessary libraries/modules
import sys
import os
import warnings
import pandas as pd
from timeit import default_timer as timer
from sklearn.linear_model import LogisticRegression

# Adding the project root to sys.path for importing custom modules from the src folder
PROJECT_ROOT = os.path.abspath(os.path.join(".."))
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# Importing specific utilities from the src package: functions for vectorization and model evaluation.
from src.evaluation import evaluate_model, get_metrics, save_metrics_to_registry  # noqa: E402
from src.preprocessing import vectorize_text  # noqa: E402

# Suppressing unnecessary warnings to keep the output more compact during training/experiments.
warnings.filterwarnings("ignore")

### 2. Feature Engineering: TF-IDF Vectorisation
To convert the cleaned tokens into numerical features, we use TF-IDF vectorisation, taking into account unigrams, bigrams and trigrams (ngram_range=(1, 3)). This will allow the model to account for common financial phrases (for example, ‘n’t work’, ‘lost my card’).

In [ ]:
# Loading the prepared datasets for training, validation, and testing.
train_df = pd.read_csv("../data/train_split.csv")
val_df = pd.read_csv("../data/val_split.csv")
test_df = pd.read_csv("../data/test_split.csv")

In [ ]:
# Vectorizing texts (TF-IDF) to obtain numerical features for training/validation/testing.
X_train_tfidf, X_val_tfidf, X_test_tfidf, vectorizer = vectorize_text(
    train_df["text"], val_df["text"], test_df["text"]
)

# Checking the size of the vocabulary — useful for controlling the complexity of the feature space.
print(f"Dictionary size: {len(vectorizer.vocabulary_)}")

Dictionary size: 4834


To check the quality of the vectorisation, we will extract random features (n-grams) from the dictionary we have created. This will allow us to verify that stemming and tokenisation have been carried out correctly and that the model is processing meaningful financial terms.

In [ ]:
# Getting the list of feature names (n-grams) from the trained TF-IDF vectorization
feature_names = vectorizer.get_feature_names_out()

# Printing sample features at regular intervals
print(f"Sample features: {feature_names[::50]}")

Sample features: ["'" "'s extra" 'abroad got' 'account explain' 'account ve' 'age limit'
 'american express' 'app direct' 'approxim' "atm machin n't" 'avail yet'
 'beneficiari possibl' "ca n't withdraw" 'card card' 'card lost'
 'card put' 'card yet' 'cash withdraw account' 'charg ad' 'charg payment'
 'check card' 'code get app' 'could get' 'currenc accept add'
 'debit account' 'delet account pleas' 'direct debit'
 'duplic charg account' 'european bank' 'exchang rate got'
 'extra charg account' 'fee cash withdraw' 'figur' 'foreign exchang rate'
 'full' 'get cash' 'get refund someth' 'go charg' 'got less money'
 'help fix' 'howev' 'interbank exchang' "know 's go" 'left phone'
 'limit much top' 'long take money' "machin n't work" 'make purchas'
 'mayb someon' 'money ask' "money n't come" 'much' "n't cash"
 "n't payment" "n't transfer account" 'need find' 'need verifi topup'
 'number dispos' 'one transact' 'paid gbp' 'payment duplic'
 'payment still pend' 'person sent money' 'pleas freez' 

### 3. Training the base model
As the classic baseline for the text multi-class classification task, we use the default Logistic Regression. We leave the regularisation parameters at their default values (C=1.0) to capture the pure capability of the linear classifier without any tuning.

In [ ]:
# Logistic Regression is chosen as the baseline due to its simplicity, high training speed, and linear interpretability of features compared to Naive Bayes or random guessing.
# We initialize and train the logistic regression; measuring the pure training time to assess the model's performance.
model_lr = LogisticRegression(max_iter=1000, random_state=42)
start_time = timer()
model_lr.fit(X_train_tfidf, train_df["label"])
pure_train_time = timer() - start_time

print(f"Training time: {pure_train_time:.2f} seconds")

Training time: 0.64 seconds


In [ ]:
# Evaluating and printing metrics to the console
evaluate_model(
    model_lr,
    X_train_tfidf,
    train_df["label"].values,
    X_val_tfidf,
    val_df["label"].values,
    X_test_tfidf,
    test_df["label"].values,
)

# Collecting metrics and saving them to the project's central registry
lr_metrics = get_metrics(
    model_lr,
    X_train_tfidf,
    train_df["label"].values,
    X_val_tfidf,
    val_df["label"].values,
    X_test_tfidf,
    test_df["label"].values,
    model_name="Logistic Regression (TF-IDF)",
    training_time=pure_train_time,
)

# Saving the obtained metrics to the experiment/results registry for further comparison
save_metrics_to_registry(lr_metrics)

ROC_AUC Train: 0.9987
ROC_AUC Val: 0.9946
ROC_AUC Test: 0.995
---
Macro F1 Train: 0.9168
Macro F1 Val: 0.8349
Macro F1 Test: 0.8341
---
Weighted F1 Train: 0.9212
Weighted F1 Val: 0.8378
Weighted F1 Test: 0.8341
 The metrics for 'Logistic Regression (TF-IDF)' have been saved to ../data/metrics_registry.csv


### 4. Baseline Results and Analysis

**Macro F1 Train: 0.9168**

**Macro F1 Val: 0.8349**

**Macro F1 Test: 0.8341**

### 💡 Key insights:
**Stability:** The model shows a minimal gap between validation and test (less than 0.1%), confirming the correctness of the data split.

**Overfitting:** The gap between training and test is around 8%. The model generalises the data well, but linear constraints prevent it from extracting more from the available 4,834 features.

**Setting the baseline result:** The value of **0.8341 Macro F1** becomes the official benchmark. Subsequent models (CatBoost, Neural Networks, BERT) must outperform this result.